# Notebook 3 — CLS Lens: Mean-Adjusted Patch Token Scoring

**What this notebook does:**
For each patch position `(i, j)` at each target ViT layer `L`:

1. Load precomputed mean embeddings (produced by `token_embedding_analysis.ipynb`):
   - `avg_cls[L]`      — mean CLS token embedding at layer L (averaged over the val set)  
   - `avg_token[L,i,j]` — mean patch token at position (i, j) and layer L
2. Apply the **mean-adjustment** to shift the patch token into the CLS distribution:
   ```
   lens_input[i,j] = h[i,j] - avg_token[L,i,j] + avg_cls[L]
   ```
3. Feed to the **CLS lens**: `lens(lens_input[i,j]) → logits`
4. `score[i,j] = softmax(logits)[y_hat]` ∈ [0, 1]

The adjustment `h - avg_token + avg_cls` removes the patch-specific mean offset and
re-centers the representation around the CLS mean, making the CLS lens's distributional
assumptions more valid than feeding raw patch tokens (Notebook 2).

All `H×W` patches are scored — no border exclusion.

Sanity-check cells (**SC N**) follow every major step.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.datasets as datasets

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from tuned_lens.model import VisionModelWrapper
from tuned_lens.config import ModelConfig
from attribute_lens.scorer import load_lens_checkpoint, discover_lens_files

print('Imports OK')

In [ ]:
# ── CONFIGURATION — edit paths to match your environment ─────────────────────
IMAGENET_VAL_DIR = '/opt/models/datasets/imagenet/extracted/val'
CLS_LENS_DIR     = '../outputs/affine_kld/best_lenses'
MEANS_PATH       = '../outputs/precomputed_means.pt'  # from token_embedding_analysis.ipynb
MODEL_NAME       = 'vit_large_patch14_clip_224.openai_ft_in1k'

TARGET_LAYERS    = [0, 6, 12, 18, 23]  # filtered to available checkpoints
N_IMAGES         = 10
SEED             = 42
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Device    : {DEVICE}')
print(f'Lens dir  : {CLS_LENS_DIR}')
print(f'Means file: {MEANS_PATH}')

## SC 1 — Verify all required paths exist

In [ ]:
assert Path(IMAGENET_VAL_DIR).exists(), f'Val dir not found: {IMAGENET_VAL_DIR}'
assert Path(CLS_LENS_DIR).exists(),     f'CLS lens dir not found: {CLS_LENS_DIR}'
assert Path(MEANS_PATH).exists(), (
    f'Means file not found: {MEANS_PATH}. '
    f'Run token_embedding_analysis.ipynb first to generate it.'
)

available_lenses = discover_lens_files(CLS_LENS_DIR)
print(f'Found {len(available_lenses)} CLS lens checkpoints')
print(f'  Available layers : {sorted(available_lenses.keys())}')

TARGET_LAYERS = sorted(l for l in TARGET_LAYERS if l in available_lenses)
assert TARGET_LAYERS, (
    f'None of the requested layers have checkpoints. '
    f'Found: {sorted(available_lenses.keys())}'
)
print(f'  Using layers     : {TARGET_LAYERS}')
print('SC 1 ✓')

## Load precomputed mean embeddings

In [ ]:
data           = torch.load(MEANS_PATH, map_location='cpu', weights_only=False)
stored_layers  = data['target_layers']   # list[int]
mean_cls_all   = data['mean_cls']        # Tensor[L, d_model]
mean_patch_all = data['mean_patch']      # Tensor[L, H, W, d_model]
patch_grid     = data['patch_grid']      # (H, W)

H_means, W_means = patch_grid
d_means = mean_cls_all.shape[1]

print(f'Means file      : {MEANS_PATH}')
print(f'Model in file   : {data.get("model_name", "unknown")}')
print(f'Images processed: {data.get("n_images", "unknown"):,}')
print(f'Stored layers   : {stored_layers}')
print(f'mean_cls  shape : {tuple(mean_cls_all.shape)}')
print(f'mean_patch shape: {tuple(mean_patch_all.shape)}')
print(f'patch_grid      : {patch_grid}')

## SC 2 — Verify mean embedding structure

In [ ]:
assert mean_cls_all.ndim == 2,    f'mean_cls should be 2D [L, d_model], got shape {mean_cls_all.shape}'
assert mean_patch_all.ndim == 4,  f'mean_patch should be 4D [L, H, W, d], got shape {mean_patch_all.shape}'
assert mean_cls_all.shape[0] == len(stored_layers), 'L dimension mismatch'
assert mean_patch_all.shape[0] == len(stored_layers), 'L dimension mismatch (patch)'
assert mean_cls_all.shape[1] == mean_patch_all.shape[3], 'd_model mismatch between cls and patch means'

# Every requested TARGET layer must be in the means file
stored_set = set(stored_layers)
for layer_idx in TARGET_LAYERS:
    assert layer_idx in stored_set, (
        f'Layer {layer_idx} not in means file. '
        f'Available: {stored_layers}. '
        f'Re-run token_embedding_analysis.ipynb with this layer.'
    )

print(f'mean_cls  : {tuple(mean_cls_all.shape)}  (L={len(stored_layers)}, d={mean_cls_all.shape[1]})')
print(f'mean_patch: {tuple(mean_patch_all.shape)}')
print(f'All TARGET_LAYERS present in means file: {TARGET_LAYERS}')
print('SC 2 ✓')

## Load model

In [ ]:
model_cfg = ModelConfig(
    model_name=MODEL_NAME,
    pretrained=True,
    target_layers=TARGET_LAYERS,
    freeze_model=True,
    patch_mode=True,   # capture patch tokens from each block
)
wrapper = VisionModelWrapper(model_cfg, device=DEVICE)

H, W = wrapper.patch_grid_size
print(f'Model      : {MODEL_NAME}')
print(f'd_model    : {wrapper.d_model}')
print(f'num_classes: {wrapper.num_classes}')
print(f'num_layers : {wrapper.num_layers}')
print(f'patch_grid : {H}x{W} = {H * W} patches per image')
print(f'target_lyrs: {wrapper.target_layers}')

## SC 3 — Cross-check model properties against means file

In [ ]:
assert wrapper.d_model > 0,                    'd_model must be positive'
assert wrapper.num_classes == 1000,             f'Expected 1000 classes, got {wrapper.num_classes}'
assert wrapper.config.patch_mode is True,       'patch_mode must be True'
assert wrapper.target_layers == TARGET_LAYERS,  'Target-layer mismatch'
assert d_means == wrapper.d_model, (
    f'means file d_model={d_means} does not match model d_model={wrapper.d_model}'
)
assert (H_means, W_means) == (H, W), (
    f'means patch_grid={patch_grid} does not match model patch_grid={(H, W)}'
)
print(f'SC 3 ✓  d_model={wrapper.d_model}, patch_grid=({H},{W}), means file consistent')

## Build per-layer mean tensors and move to device

In [ ]:
layer_to_pos = {l: i for i, l in enumerate(stored_layers)}

mean_cls_per_layer   = {}  # {layer_idx: Tensor[d_model]}      on DEVICE
mean_patch_per_layer = {}  # {layer_idx: Tensor[H, W, d_model]} on DEVICE

for layer_idx in TARGET_LAYERS:
    pos = layer_to_pos[layer_idx]
    mean_cls_per_layer[layer_idx]   = mean_cls_all[pos].to(DEVICE)
    mean_patch_per_layer[layer_idx] = mean_patch_all[pos].to(DEVICE)   # [H, W, d_model]

    mc = mean_cls_per_layer[layer_idx]
    mp = mean_patch_per_layer[layer_idx]
    print(
        f'  Layer {layer_idx:2d}:  '
        f'mean_cls norm={mc.norm():.3f}  '
        f'mean_patch avg_norm={mp.norm(dim=-1).mean():.3f}'
    )

print('Per-layer means ready.')

## SC 4 — Verify per-layer mean tensor shapes and norms

In [ ]:
for layer_idx in TARGET_LAYERS:
    mc = mean_cls_per_layer[layer_idx]
    mp = mean_patch_per_layer[layer_idx]

    assert mc.shape == (wrapper.d_model,), (
        f'Layer {layer_idx}: mean_cls shape {mc.shape}, expected ({wrapper.d_model},)'
    )
    assert mp.shape == (H, W, wrapper.d_model), (
        f'Layer {layer_idx}: mean_patch shape {mp.shape}, expected ({H},{W},{wrapper.d_model})'
    )
    assert mc.norm().item() > 0, f'Layer {layer_idx}: mean_cls is all-zero — unexpected'
    assert not torch.isnan(mc).any(), f'Layer {layer_idx}: NaN in mean_cls'
    assert not torch.isnan(mp).any(), f'Layer {layer_idx}: NaN in mean_patch'

    print(f'  Layer {layer_idx:2d}: mean_cls {tuple(mc.shape)}  mean_patch {tuple(mp.shape)}  ✓')

print('SC 4 ✓')

## Load ImageNet val dataset and sample 10 random images

In [ ]:
transform   = wrapper.get_transform()
dataset     = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=transform)
raw_dataset = datasets.ImageFolder(IMAGENET_VAL_DIR)  # no transform — PIL for visualization
class_names = dataset.classes

rng            = random.Random(SEED)
sample_indices = rng.sample(range(len(dataset)), N_IMAGES)

print(f'Val images : {len(dataset):,}  ({len(class_names)} classes)')
print(f'Sampled idx: {sample_indices}')

## SC 5 — Verify dataset loading and display sampled images

In [ ]:
img_chk, lbl_chk = dataset[sample_indices[0]]
assert img_chk.shape == (3, 224, 224), f'Expected (3,224,224), got {img_chk.shape}'
assert 0 <= lbl_chk < len(class_names), f'Label {lbl_chk} out of range'
print(f'Tensor shape: {tuple(img_chk.shape)}   label: {lbl_chk} ({class_names[lbl_chk]})')

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, idx in zip(axes.flat, sample_indices):
    pil_img, lbl = raw_dataset[idx]
    ax.imshow(pil_img.resize((224, 224)))
    ax.set_title(f'{class_names[lbl][:18]}\nidx={idx}', fontsize=8)
    ax.axis('off')
plt.suptitle('10 Sampled ImageNet Val Images', fontsize=13)
plt.tight_layout()
plt.show()
print('SC 5 ✓')

## Extract patch tokens for all sampled images

In [ ]:
all_patch_states = {}  # {img_idx: {layer_idx: Tensor[1, H, W, d_model]}}
all_logits       = {}  # {img_idx: Tensor[1, num_classes]}
all_y_hats       = {}  # {img_idx: int}

for n, idx in enumerate(sample_indices):
    img_t, _ = dataset[idx]
    batch     = img_t.unsqueeze(0).to(DEVICE)        # [1, 3, 224, 224]

    patch_states, logits = wrapper.extract_patches(batch)

    all_patch_states[idx] = {k: v.cpu() for k, v in patch_states.items()}
    all_logits[idx]       = logits.cpu()
    all_y_hats[idx]       = int(logits[0].argmax())

    pred = class_names[all_y_hats[idx]]
    print(f'  [{n+1:2d}/{N_IMAGES}] idx={idx:6d}  y_hat={all_y_hats[idx]:4d}  ({pred})')

print('Extraction complete.')

## SC 6 — Verify patch tensor shapes and value ranges

In [ ]:
idx0 = sample_indices[0]
print(f'Checking shapes for image idx={idx0}:')
for layer_idx in TARGET_LAYERS:
    ps  = all_patch_states[idx0][layer_idx]
    exp = (1, H, W, wrapper.d_model)
    assert ps.shape == exp, f'Layer {layer_idx}: expected {exp}, got {ps.shape}'
    print(f'  Layer {layer_idx:2d}: shape={tuple(ps.shape)}  range=[{ps.min():.3f}, {ps.max():.3f}]  ✓')

lg = all_logits[idx0]
assert lg.shape == (1, wrapper.num_classes), f'Logits shape: {lg.shape}'
print(f'\nLogits shape  : {tuple(lg.shape)}')
print(f'y_hat for idx0: {all_y_hats[idx0]}  ({class_names[all_y_hats[idx0]]})')
print('SC 6 ✓')

## Load CLS lens checkpoints

In [ ]:
lenses = {}   # {layer_idx: BaseLens}
for layer_idx in TARGET_LAYERS:
    lenses[layer_idx] = load_lens_checkpoint(available_lenses[layer_idx], device=DEVICE)
    print(f'  Layer {layer_idx:2d}: {type(lenses[layer_idx]).__name__}')
print('CLS lenses loaded.')

## SC 7 — Verify CLS lens input/output dimensions

In [ ]:
expected_in  = wrapper.d_model    # mean-adjusted token is still d_model dimensional
expected_out = wrapper.num_classes
print(f'Expected lens input  = d_model     = {expected_in}')
print(f'Expected lens output = num_classes = {expected_out}\n')

for layer_idx, lens in lenses.items():
    if hasattr(lens, 'linear'):   # AffineLens
        w_in  = lens.linear.weight.shape[1]
        w_out = lens.linear.weight.shape[0]
    else:                          # MLPLens
        lins  = [m for m in lens.net.modules() if hasattr(m, 'weight') and m.weight.dim() == 2]
        w_in  = lins[0].weight.shape[1]
        w_out = lins[-1].weight.shape[0]

    in_ok  = 'OK' if w_in  == expected_in  else f'MISMATCH (got {w_in})'
    out_ok = 'OK' if w_out == expected_out else f'MISMATCH (got {w_out})'
    print(f'  Layer {layer_idx:2d}: in={w_in} [{in_ok}]   out={w_out} [{out_ok}]')
    assert w_in  == expected_in,  f'Layer {layer_idx} input dim mismatch'
    assert w_out == expected_out, f'Layer {layer_idx} output dim mismatch'

print('\nSC 7 ✓')

## SC 8 — Sanity-check the mean adjustment formula

For a single patch, verify: `adjusted = h - mean_patch + mean_cls` has the correct shape
and that the adjustment is non-trivial (the vectors are not identical).

In [ ]:
idx0       = sample_indices[0]
layer0     = TARGET_LAYERS[0]
pi, pj     = 8, 8

# Raw patch token
h_ij    = all_patch_states[idx0][layer0][0, pi, pj, :].to(DEVICE)   # [d_model]

# Mean vectors
mc      = mean_cls_per_layer[layer0]           # [d_model]
mp_ij   = mean_patch_per_layer[layer0][pi, pj] # [d_model]

# Adjustment
adjusted = h_ij - mp_ij + mc                  # [d_model]

print(f'h[{pi},{pj}]  shape : {tuple(h_ij.shape)}')
print(f'mean_cls      shape : {tuple(mc.shape)}')
print(f'mean_patch[{pi},{pj}] shape : {tuple(mp_ij.shape)}')
print(f'adjusted      shape : {tuple(adjusted.shape)}')

assert adjusted.shape == (wrapper.d_model,), f'Adjusted shape mismatch: {adjusted.shape}'
diff_from_raw = (adjusted - h_ij).norm().item()
print(f'\n||adjusted - raw||_2 = {diff_from_raw:.4f}  (should be > 0 unless mean_cls == mean_patch)')
assert diff_from_raw > 0, 'Adjustment made no change — mean_cls == mean_patch[i,j] for this position'
print('SC 8 ✓')

## Compute mean-adjusted score maps

For each patch `(i, j)` at layer `L`:
```
adjusted[i,j] = h[i,j] - avg_token[L,i,j] + avg_cls[L]
score[i,j]    = softmax(cls_lens_L(adjusted[i,j]))[y_hat]
```

In [ ]:
all_score_maps = {}  # {img_idx: {layer_idx: Tensor[H, W]}}

for img_idx in sample_indices:
    all_score_maps[img_idx] = {}
    y_hat = all_y_hats[img_idx]

    for layer_idx in TARGET_LAYERS:
        patches = all_patch_states[img_idx][layer_idx].to(DEVICE)   # [1, H, W, d]
        _, H_p, W_p, d = patches.shape

        flat    = patches[0].reshape(H_p * W_p, d)                    # [H*W, d_model]
        mp_flat = mean_patch_per_layer[layer_idx].reshape(H_p * W_p, d)  # [H*W, d_model]
        mc      = mean_cls_per_layer[layer_idx]                        # [d_model]

        # Mean-adjustment: remove patch position bias, re-center around CLS distribution
        adjusted = flat - mp_flat + mc.unsqueeze(0)                   # [H*W, d_model]

        with torch.no_grad():
            out_logits = lenses[layer_idx](adjusted)                  # [H*W, num_classes]
        scores = F.softmax(out_logits, dim=-1)[:, y_hat].cpu()        # [H*W]

        all_score_maps[img_idx][layer_idx] = scores.reshape(H_p, W_p)

print('Mean-adjusted scoring complete for all images and layers.')

## SC 9 — Verify score map shapes, absence of NaN, and value ranges

In [ ]:
idx0 = sample_indices[0]
print(f'Checking score maps for image idx={idx0}:')
for layer_idx in TARGET_LAYERS:
    sm = all_score_maps[idx0][layer_idx]
    assert sm.shape == (H, W), f'Layer {layer_idx}: shape {sm.shape}'
    assert not torch.isnan(sm).any(), f'Layer {layer_idx}: unexpected NaN values'
    assert (sm >= 0.0).all() and (sm <= 1.0).all(), (
        f'Layer {layer_idx}: scores out of [0,1] — [{sm.min():.4f}, {sm.max():.4f}]'
    )
    print(
        f'  Layer {layer_idx:2d}  all {H*W} patches scored  '
        f'range=[{sm.min():.4f},{sm.max():.4f}]  mean={sm.mean():.4f}  ✓'
    )

print('\nSC 9 ✓')

## SC 10 — Manual spot-check: recompute one patch score by hand

In [ ]:
idx0   = sample_indices[0]
layer0 = TARGET_LAYERS[0]
y_hat0 = all_y_hats[idx0]
pi, pj = 8, 8

h_ij    = all_patch_states[idx0][layer0][0, pi, pj, :].to(DEVICE)
mc      = mean_cls_per_layer[layer0]
mp_ij   = mean_patch_per_layer[layer0][pi, pj]
adj     = h_ij - mp_ij + mc                            # [d_model]

with torch.no_grad():
    manual_logits = lenses[layer0](adj.unsqueeze(0))   # [1, C]
manual_score  = float(F.softmax(manual_logits, dim=-1)[0, y_hat0])
stored_score  = float(all_score_maps[idx0][layer0][pi, pj])

print(f'Manual score  for patch ({pi},{pj}) layer {layer0}: {manual_score:.6f}')
print(f'Stored score  for patch ({pi},{pj}) layer {layer0}: {stored_score:.6f}')
assert abs(manual_score - stored_score) < 1e-5, (
    f'Mismatch: manual={manual_score:.6f}, stored={stored_score:.6f}'
)
print('SC 10 ✓  Manual and stored scores match')

## Visualize: score grid + heatmap overlay on original image

In [ ]:
for img_idx in sample_indices:
    y_hat       = all_y_hats[img_idx]
    pil_img, _  = raw_dataset[img_idx]
    img_np      = np.array(pil_img.resize((224, 224)))

    n_cols = len(TARGET_LAYERS) + 1
    fig, axes = plt.subplots(2, n_cols, figsize=(3.6 * n_cols, 8))

    # ── Column 0: original image ───────────────────────────────────────────
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title(f'Original  idx={img_idx}', fontsize=8)
    axes[0, 0].axis('off')
    axes[1, 0].text(
        0.5, 0.5, f'Pred:\n{class_names[y_hat][:22]}',
        ha='center', va='center', fontsize=8,
        transform=axes[1, 0].transAxes
    )
    axes[1, 0].axis('off')

    # ── Columns 1..N: per-layer score grid + overlay ───────────────────────
    for col, layer_idx in enumerate(TARGET_LAYERS, start=1):
        sm      = all_score_maps[img_idx][layer_idx].numpy()
        vmax_sm = float(sm.max()) if sm.max() > 1e-8 else 1.0

        # Top row: score grid at patch resolution
        im = axes[0, col].imshow(sm, cmap='hot', vmin=0.0, vmax=vmax_sm)
        axes[0, col].set_title(f'Layer {layer_idx}  Score Grid ({H}x{W})', fontsize=8)
        axes[0, col].axis('off')
        plt.colorbar(im, ax=axes[0, col], fraction=0.046, pad=0.04)

        # Bottom row: normalized heatmap overlay
        hm_norm = (sm - sm.min()) / (sm.max() - sm.min() + 1e-8)
        hm_u8   = (hm_norm * 255).astype(np.uint8)
        hm_224  = np.array(Image.fromarray(hm_u8).resize((224, 224), Image.NEAREST))
        axes[1, col].imshow(img_np)
        axes[1, col].imshow(hm_224, cmap='hot', alpha=0.55, vmin=0, vmax=255)
        axes[1, col].set_title(f'Layer {layer_idx}  Overlay', fontsize=8)
        axes[1, col].axis('off')

    title = f'CLS Lens (mean-adjusted)  idx={img_idx}  Pred: {class_names[y_hat]}'
    plt.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()